# 🛡️ FYRP: Explainable Multi-Agent Adversarial Threat Detection
## 📋 Comprehensive Project Walkthrough

This notebook is an interactive, step-by-step guide to our custom adversarial threat detection pipeline for facial biometrics. 

### Why Custom Implementation over Libraries (like ART)?
Generic libraries like the Adversarial Robustness Toolbox (ART) are excellent for quick testing on datasets like CIFAR-10. However, for a final year research project focusing on **Explainable AI (XAI)**, using a "black-box" wrapper limits your ability to explain the underlying mathematics to examiners. 

In this notebook, we demonstrate our transparent, PyTorch-native implementation covering:
1. **Face-Specific Preprocessing**
2. **Mathematical implementation of FGSM / PGD attacks**
3. **The Novel Dynamic Threshold Engine**
4. **LLM-based Advocate & Judge Agents**
5. **Grad-CAM Visualizations**

---
## 📚 Step 1: Initialization & Imports
We load PyTorch, Matplotlib for visualization, and our custom modules.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os

# Import custom FYRP modules
from model.config import Config
from model.cnn_model import AdversarialFaceDetector, FeatureExtractor
from model.adversarial import FGSM
from model.threshold import DynamicThresholdEngine
from utils.preprocessing import preprocess_image
from agents import run_advocates, JudgeAgent, format_debate_for_judge

# Set device
device = Config.DEVICE if torch.cuda.is_available() else "cpu"
print(f"🔧 Using device: {device}")

---
## 🖼️ Step 2: Loading & Visualizing a Sample Face
We use a standard sample image to demonstrate the pipeline. Our CNN expects 224x224 normalized images.

In [ ]:
# Load sample image
img_path = "images.jpg" # Make sure this file exists in your directory
if os.path.exists(img_path):
    tensor_img = preprocess_image(img_path).to(device)
    
    # Denormalize for plotting
    mean = torch.tensor(Config.NORMALIZE_MEAN).view(3, 1, 1).to(device)
    std = torch.tensor(Config.NORMALIZE_STD).view(3, 1, 1).to(device)
    display_img = (tensor_img.squeeze(0) * std + mean).clamp(0, 1).permute(1, 2, 0).cpu().numpy()
    
    plt.figure(figsize=(4,4))
    plt.imshow(display_img)
    plt.title("Original Clean Face")
    plt.axis("off")
    plt.show()
else:
    print(f"Sample image '{img_path}' not found. Please provide an image to visualize.")

---
## 🧠 Step 3: Loading the Trained CNN
We load our trained `AdversarialFaceDetector` (ResNet-18 backbone) which was trained in Phase 1 (Clean) and Phase 2 (Adversarial).

In [ ]:
# Load the FeatureExtractor wrapper which handles the model and checkpoints
checkpoint_path = "checkpoints/best_model.pth"

if os.path.exists(checkpoint_path):
    extractor = FeatureExtractor(checkpoint_path, device=device)
    print("✅ Best model checkpoint loaded successfully.")
    
    # Run a forward pass
    features = extractor.extract(tensor_img)
    print("\n📊 Extracted Features:")
    for k, v in features.items():
        if k != "embedding":
            print(f"  {k}: {v}")
else:
    print(f"Checkpoint not found at {checkpoint_path}. Please run training first.")

---
## ⚔️ Step 4: Transparent Adversarial Attack (FGSM)
Instead of using `ART`, we manually compute the gradient of the loss with respect to the input image. This shows exactly how an attacker manipulates the pixels.

**Formula:** $ x_{adv} = x + \epsilon \cdot 	ext{sign}(\nabla_x L) $

In [ ]:
# Instantiate our custom FGSM attacker
epsilon = 0.05 # Perturbation budget
fgsm_attacker = FGSM(extractor.model, epsilon=epsilon)

# True label for a clean image is 0 ("normal")
true_label = torch.tensor([0]).to(device)

# Generate attack
adv_tensor = fgsm_attacker.perturb(tensor_img, true_label)

# Run the attacked image through the model
adv_features = extractor.extract(adv_tensor)

print(f"🎯 Prediction changed from {features['prediction'].upper()} to {adv_features['prediction'].upper()}")

# Visual comparison
adv_display = (adv_tensor.squeeze(0) * std + mean).clamp(0, 1).permute(1, 2, 0).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(display_img)
axes[0].set_title(f"Clean Image\nPred: {features['prediction']}")
axes[0].axis('off')

axes[1].imshow(adv_display)
axes[1].set_title(f"Adversarial Image (FGSM eps={epsilon})\nPred: {adv_features['prediction']}")
axes[1].axis('off')
plt.show()

---
## ⚖️ Step 5: The Dynamic Threshold Engine
Static thresholds fail in real-world deployments. We compute a context-aware threshold based on Confidence, Anomaly Score, Embedding Drift, and Image Quality.

In [ ]:
engine = DynamicThresholdEngine()

# Compute threshold decision for the adversarial image
decision = engine.compute(adv_features, adv_tensor)

print("📝 Threshold Engine Decision:")
print(f"Raw CNN Label : {decision.raw_prediction}")
print(f"Final Label   : {decision.final_label}")
print(f"Risk Level    : {decision.risk_level}")
print(f"Computed Threshold (T) : {decision.computed_threshold:.4f}")
print("\nFactors:")
print(f" - Confidence (CNN) : {decision.confidence_score:.4f}")
print(f" - Anomaly Score    : {decision.anomaly_score:.4f}")

---
## 🧑‍⚖️ Step 6: Multi-Agent Debate System
To prevent automated systems from "hallucinating" or making overly rigid decisions, we pass the data to LLM-powered agents.
- **Proponent Advocate:** Argues the image is safe.
- **Opponent Advocate:** Argues the image is a threat.
- **Judge Agent:** Evaluates the debate and technical data to issue a final verdict.

In [ ]:
# Format data for agents
cnn_str = f"Prediction: {adv_features['prediction']}, Confidence: {adv_features['confidence_score']}"
threshold_str = engine.format_for_agent(decision)

print("⏳ Running Advocates (This connects to the LLM API...)")
try:
    pro_out, opp_out = run_advocates(cnn_str, threshold_str)
    
    print("\n🟢 PROPONENT ARGUMENT:")
    print(pro_out.content)
    
    print("\n🔴 OPPONENT ARGUMENT:")
    print(opp_out.content)
    
    # Run Judge
    print("\n⏳ Running Judge...")
    transcript = format_debate_for_judge(pro_out, opp_out)
    judge = JudgeAgent()
    verdict = judge.verdict(transcript, threshold_str)
    
    print("\n⚖️ FINAL VERDICT:")
    print(f"Decision: {verdict.final_decision} (Risk: {verdict.risk_level})")
    print(f"Reasoning: {verdict.reasoning}")
except Exception as e:
    print(f"API Error (Check .env configuration): {e}")

---
## 🗺️ Step 7: XAI - Grad-CAM Visualization
Finally, we generate a Grad-CAM heatmap to show exactly which pixels influenced the CNN's decision. This provides crucial spatial explainability.

In [ ]:
from utils.visualize import apply_gradcam

try:
    print("Generating Grad-CAM heatmap...")
    # apply_gradcam expects file paths, so we use the original image path
    fig = apply_gradcam(extractor.model, img_path, target_layer="layer4", device=device)
    plt.show()
except Exception as e:
    print(f"Grad-CAM Error: {e}")